In [ ]:
import copy
import torch


# ==========================================================
# Base Callback
# ==========================================================

class Callback:

    def on_train_begin(self, trainer):
        pass

    def on_epoch_begin(self, trainer):
        pass

    def on_epoch_end(self, trainer):
        pass

    def on_train_end(self, trainer):
        pass


# ==========================================================
# Early Stopping Callback
# ==========================================================

class EarlyStopping(Callback):

    def __init__(self, patience=20):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0

    def on_epoch_end(self, trainer):
        if trainer.val_loss < self.best_loss:
            self.best_loss = trainer.val_loss
            self.counter = 0
            trainer.best_state = copy.deepcopy(
                trainer.model.state_dict()
            )

        else:
            self.counter += 1

        if self.counter >= self.patience:
            trainer.stop_training = True


# ==========================================================
# Progress Printer
# ==========================================================

class ProgressPrinter(Callback):

    def on_epoch_end(self, trainer):
        print(
            f"Epoch {trainer.epoch:4d} | "
            f"Train Loss = {trainer.train_loss:.4f} | "
            f"Val Loss = {trainer.val_loss:.4f}"
        )


# ==========================================================
# Loss History
# ==========================================================

class LossHistory(Callback):

    def __init__(self):
        self.train_losses = []
        self.val_losses = []

    def on_epoch_end(self, trainer):

        self.train_losses.append(trainer.train_loss)
        self.val_losses.append(trainer.val_loss)


# ==========================================================
# Trainer
# ==========================================================

class Trainer:

    def __init__(
        self,
        model,
        optimizer,
        criterion,
        callbacks=None,
    ):

        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion

        self.callbacks = callbacks or [] 
        self.stop_training = False
        self.best_state = None

        self.epoch = 0
        self.train_loss = None
        self.val_loss = None

    def fit(self, Xtrain, ytrain, Xval, yval, epochs):

        for cb in self.callbacks:
            cb.on_train_begin(self)

        for epoch in range(epochs):

            self.epoch = epoch

            for cb in self.callbacks:
                cb.on_epoch_begin(self)

            # ------------------------
            # Training
            # ------------------------

            self.model.train()
            self.optimizer.zero_grad()
            y_pred = self.model(Xtrain)
            loss = self.criterion(y_pred, ytrain)
            loss.backward()
            self.optimizer.step()
            self.train_loss = loss.item()

            # ------------------------
            # Validation
            # ------------------------

            self.model.eval()

            with torch.no_grad():
                y_val_pred = self.model(Xval)
                self.val_loss = self.criterion(
                    y_val_pred,
                    yval
                ).item()

            # ------------------------
            # Callbacks
            # ------------------------

            for cb in self.callbacks:
                cb.on_epoch_end(self)

            if self.stop_training:
                print(f"Early stopping at epoch {epoch}")
                break

        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)

        for cb in self.callbacks:
            cb.on_train_end(self)


# ==========================================================
# Usage
# ==========================================================

history = LossHistory()

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    callbacks=[
        ProgressPrinter(),
        history,
        EarlyStopping(patience=20),
    ],
)

trainer.fit(
    Xtrain,
    ytrain,
    Xval,
    yval,
    epochs=1000,
)

# Loss curves are available afterwards

train_losses = history.train_losses
val_losses = history.val_losses

In [ ]:
from abc import ABC
from dataclasses import dataclass

import copy
import torch
from torch import nn


# ==========================================================
# Callback Context
# ==========================================================
# Small shared object passed to callbacks instead of the full Trainer.

@dataclass
class CallbackContext:
    model: nn.Module
    epoch: int
    train_loss: float
    val_loss: float
    stop_training: bool = False
    best_state: dict | None = None


# ==========================================================
# Base Callback (no-op hooks)
# ==========================================================

class Callback(ABC):

    def on_train_begin(self, ctx):
        return

    def on_epoch_begin(self, ctx):
        return

    def on_epoch_end(self, ctx):
        return

    def on_train_end(self, ctx):
        return


# ==========================================================
# Concrete Callbacks
# ==========================================================

class ProgressPrinter(Callback):

    def on_epoch_end(self, ctx):
        print(
            f"Epoch {ctx.epoch:4d} | "
            f"Train Loss = {ctx.train_loss:.4f} | "
            f"Val Loss = {ctx.val_loss:.4f}"
        )


class LossHistory(Callback):

    def __init__(self):
        self.train_losses = []
        self.val_losses = []

    def on_epoch_end(self, ctx):
        self.train_losses.append(ctx.train_loss)
        self.val_losses.append(ctx.val_loss)


class EarlyStopping(Callback):

    def __init__(
        self,
        patience=20,
        min_delta=0.0,
        mode="min",  # "min" for loss, "max" for metrics
        restore_best_weights=True,
    ):
        if mode not in {"min", "max"}:
            raise ValueError("mode must be 'min' or 'max'")

        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.restore_best_weights = restore_best_weights

        self.best_score = float("inf") if mode == "min" else float("-inf")
        self.counter = 0

    def _improved(self, value):
        if self.mode == "min":
            return value < (self.best_score - self.min_delta)
        return value > (self.best_score + self.min_delta)

    def on_epoch_end(self, ctx):
        current = ctx.val_loss

        if self._improved(current):
            self.best_score = current
            self.counter = 0
            if self.restore_best_weights:
                ctx.best_state = copy.deepcopy(ctx.model.state_dict())
        else:
            self.counter += 1

        if self.counter >= self.patience:
            ctx.stop_training = True


# ==========================================================
# Trainer
# ==========================================================

class Trainer:

    def __init__(
        self,
        model,
        optimizer,
        criterion,
        callbacks=None,
    ):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.callbacks = list(callbacks) if callbacks is not None else []

    def fit(
        self,
        Xtrain,
        ytrain,
        Xval,
        yval,
        epochs,
    ):
        ctx = CallbackContext(
            model=self.model,
            epoch=0,
            train_loss=float("nan"),
            val_loss=float("nan"),
        )

        for cb in self.callbacks:
            cb.on_train_begin(ctx)

        for epoch in range(epochs):
            ctx.epoch = epoch

            for cb in self.callbacks:
                cb.on_epoch_begin(ctx)

            self.model.train()
            self.optimizer.zero_grad()
            y_pred = self.model(Xtrain)
            train_loss = self.criterion(y_pred, ytrain)
            train_loss.backward()
            self.optimizer.step()
            ctx.train_loss = float(train_loss.item())

            self.model.eval()
            with torch.no_grad():
                y_val_pred = self.model(Xval)
                val_loss = self.criterion(y_val_pred, yval)
                ctx.val_loss = float(val_loss.item())

            for cb in self.callbacks:
                cb.on_epoch_end(ctx)

            if ctx.stop_training:
                print(f"Early stopping at epoch {epoch}")
                break

        if ctx.best_state is not None:
            self.model.load_state_dict(ctx.best_state)

        for cb in self.callbacks:
            cb.on_train_end(ctx)

        return ctx


# ==========================================================
# Usage
# ==========================================================
# Replace these placeholders with your actual model/data.
# model = ...
# optimizer = ...
# criterion = ...
# Xtrain, ytrain, Xval, yval = ...

history = LossHistory()
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    callbacks=[
        ProgressPrinter(),
        history,
        EarlyStopping(
            patience=20,
            min_delta=1e-4,
            mode="min",
            restore_best_weights=True,
        ),
    ],
)

ctx = trainer.fit(Xtrain, ytrain, Xval, yval, epochs=1000)

train_losses = history.train_losses
val_losses = history.val_losses